# 🚀 Sentinel 10-Class Practical Indian Traffic AI — Cloud GPU Training Suite
### End-to-End Google Colab & Kaggle Training Pipeline for High-Accuracy Gujarat Police CCTV Detections

This notebook trains an industrial-grade **10-Class Indian Road Object Detector** optimized for high-angle surveillance cameras, night sodium vapor illumination, and intense traffic junction density.

---
### 🎯 The 10 Practical Classes:
1. `0: pedestrian` — Foot travelers, zebra crossings, vendors
2. `1: car` — Sedans, hatchbacks, SUVs, private motorcars
3. `2: two_wheeler` — Motorcycles, scooters, mopeds, bicycles
4. `3: heavy_machinery` — Tractors, JCBs, excavators, cranes, road rollers
5. `4: emergency_vehicle` — Ambulances, police patrol PCRs, fire tenders
6. `5: van` — Omni, Eeco, Tempo Travelers, commercial delivery vans
7. `6: truck` — Freight lorries, dumpers, container multi-axle trucks
8: `7: bus` — State transit GSRTC, private coaches, city transit buses
9: `8: auto_rickshaw` — 3-wheel passenger autos, e-rickshaws, chhakdas
10: `9: others` — Bullock/animal carts, handcarts, specialized vehicles

In [ ]:
# [Cell 1] Verify NVIDIA GPU Environment (T4 / V100 / A100 / L4)
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Capacity: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

In [ ]:
# [Cell 2] Install Ultra-Modern Vision Stack
!pip install -q ultralytics albumentations pyyaml matplotlib seaborn opencv-python

In [ ]:
# [Cell 3] Optional Google Drive Mount for Persistent Checkpoint Storage
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BACKUP_DIR = '/content/drive/MyDrive/Sentinel_10Class_Checkpoints'
    os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
    print(f"✅ Google Drive Connected! Checkpoints will persist at: {DRIVE_BACKUP_DIR}")
except Exception as e:
    DRIVE_BACKUP_DIR = None
    print("ℹ️ Running standalone without Google Drive. Checkpoints saved locally.")

In [ ]:
# [Cell 4] Unzip & Inspect Sentinel 10-Class Dataset
import zipfile
import yaml
import glob

zip_candidates = [
    "SENTINEL_10CLASS_GUJARAT_TRAFFIC_DATASET.zip",
    "/content/SENTINEL_10CLASS_GUJARAT_TRAFFIC_DATASET.zip",
    "/content/drive/MyDrive/SENTINEL_10CLASS_GUJARAT_TRAFFIC_DATASET.zip"
]
chosen_zip = next((z for z in zip_candidates if os.path.exists(z)), None)

dataset_dir = "/content/sentinel_10class_gujarat_dataset"
if chosen_zip:
    print(f"📦 Extracting {chosen_zip} to {dataset_dir}...")
    with zipfile.ZipFile(chosen_zip, 'r') as z:
        z.extractall("/content/")
    print("✅ Extraction Complete!")
else:
    print("⚠️ Zip archive not found locally. Please upload SENTINEL_10CLASS_GUJARAT_TRAFFIC_DATASET.zip to Colab.")

# Locate data.yaml
yaml_path = "/content/datasets/sentinel_10class_gujarat_dataset/data.yaml"
if not os.path.exists(yaml_path):
    yaml_path = glob.glob("/content/**/data.yaml", recursive=True)[0]

with open(yaml_path, 'r') as f:
    cfg = yaml.safe_load(f)

print(f"\n📊 Dataset Configuration from {yaml_path}:")
print(f"Number of Classes: {cfg.get('nc')}")
for idx, name in cfg.get('names', {}).items():
    print(f"  Class {idx}: {name}")

In [ ]:
# [Cell 5] Start 10-Class High-Performance YOLO Training
from ultralytics import YOLO

# Select base architecture: 'yolo12s.pt' or 'yolo12n.pt' or 'yolov8m.pt'
model = YOLO('yolo12s.pt')

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    # Augmentations customized for Gujarat multi-junction conditions
    mosaic=1.0,
    mixup=0.15,
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    degrees=5.0,
    scale=0.35,
    fliplr=0.5,
    project='/content/runs/train',
    name='sentinel_10class_cloud',
    exist_ok=True,
    patience=12,
    verbose=True
)

print("🎉 Training Complete!")

In [ ]:
# [Cell 6] Validate Model & Compute Per-Class Precision, Recall, mAP50
best_model_path = '/content/runs/train/sentinel_10class_cloud/weights/best.pt'
val_model = YOLO(best_model_path)
metrics = val_model.val(data=yaml_path, split='val')

print(f"\n🏆 Overall mAP@0.5: {metrics.box.map50 * 100:.2f}%")
print(f"🏆 Overall mAP@0.5:0.95: {metrics.box.map * 100:.2f}%")

print("\n📈 Per-Class Performance:")
for i, cname in val_model.names.items():
    p = metrics.box.p[i] * 100
    r = metrics.box.r[i] * 100
    m50 = metrics.box.maps[i] * 100
    print(f"  {cname.ljust(18)} | Precision: {p:5.1f}% | Recall: {r:5.1f}% | mAP50: {m50:5.1f}%")

In [ ]:
# [Cell 7] Export Model to Production Formats (ONNX & TensorRT)
import shutil

# 1. Export ONNX for high-speed cross-platform inference
val_model.export(format='onnx', dynamic=True, simplify=True)
print("✅ Exported ONNX model!")

# 2. Save best.pt backup to Google Drive if connected
if DRIVE_BACKUP_DIR and os.path.exists(DRIVE_BACKUP_DIR):
    shutil.copy2(best_model_path, os.path.join(DRIVE_BACKUP_DIR, 'sentinel_10class_best.pt'))
    print(f"✅ Backed up weights to Google Drive: {DRIVE_BACKUP_DIR}/sentinel_10class_best.pt")